In [55]:
# from https://colab.research.google.com/drive/1ctgygDRJhVGUJTQy8-bRZCl1WNcT8De6?usp=sharing#scrollTo=WtvQLpd2MCi5
# and https://github.com/lm-sys/FastChat/blob/main/fastchat/llm_judge/README.md
from datasets import load_dataset
import pandas as pd

In [87]:
dataset = load_dataset("lmsys/mt_bench_human_judgments")

dfs = []
for split_name, split in dataset.items():
    df = split.to_pandas()
    df["grader"] = split_name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

def conv_length(conv):
    return sum(len(turn["content"]) for turn in conv)

# Remove ties for simplicity
df = df_all[df_all['winner'].isin(['model_a', 'model_b'])].copy()

print(f"Rows after filtering: {len(df)}")

# Processing
df["len_a"] = df["conversation_a"].apply(conv_length)
df["len_b"] = df["conversation_b"].apply(conv_length)
df["length_diff"] = df["len_a"] - df["len_b"]

# Create llm_pair (alphabetically sorted for canonical comparison)
df['llm_pair'] = df.apply(
    lambda row: '_vs_'.join(sorted([row['model_a'], row['model_b']])),
    axis=1
)

# Add individual model columns
df['left_model'] = df['model_a']   # Model in position A (left)
df['right_model'] = df['model_b']  # Model in position B (right)

# Create position variable (which model is on left/A position)
def get_position(row):
    """Returns 'left' if first alphabetical model is in position A, else 'right'"""
    models_sorted = sorted([row['model_a'], row['model_b']])
    return 'left' if row['model_a'] == models_sorted[0] else 'right'

df['position'] = df.apply(get_position, axis=1)

# Numeric position for modeling (-0.5 = left, +0.5 = right)
df['position_numeric'] = df['position'].map({'left': -0.5, 'right': 0.5})

# Outcome: was left side (model_a) chosen?
df["left_chosen"] = (df["winner"] == "model_a").astype(int)

# Drop dict columns
df = df.drop(columns=["conversation_a", "conversation_b"])

# Verify data structure
print("\n=== Data Structure Check ===")
print(f"Unique models: {df['left_model'].nunique()} (also {df['right_model'].nunique()})")
print(f"Models: {sorted(df['left_model'].unique())}")
print(f"\nUnique llm pairs: {df['llm_pair'].nunique()}")
print(f"\nUnique questions: {df['question_id'].nunique()}")
print(f"\nPosition distribution:")
print(df['position'].value_counts())
print(f"\nPosition numeric distribution:")
print(df['position_numeric'].value_counts())

# Verify counterbalancing
print(f"\nLeft chosen distribution:")
print(df['left_chosen'].value_counts())
print(f"Left chosen rate: {df['left_chosen'].mean():.3f}")

# Save to jsonl
df.to_json("data/all.jsonl", orient="records", lines=True)

print(f"\nSaved {len(df)} rows to data/all.jsonl")

Rows after filtering: 4375

=== Data Structure Check ===
Unique models: 6 (also 6)
Models: ['alpaca-13b', 'claude-v1', 'gpt-3.5-turbo', 'gpt-4', 'llama-13b', 'vicuna-13b-v1.2']

Unique llm pairs: 15

Unique questions: 80

Position distribution:
position
right    2424
left     1951
Name: count, dtype: int64

Position numeric distribution:
position_numeric
 0.5    2424
-0.5    1951
Name: count, dtype: int64

Left chosen distribution:
left_chosen
0    2892
1    1483
Name: count, dtype: int64
Left chosen rate: 0.339

Saved 4375 rows to data/all.jsonl
